# Img2GPS — Walkthrough

CIS 5190 final project, Track A. We predict GPS coordinates from a single image taken on Penn's campus (test rectangle: 33rd & Walnut → 34th & Spruce). The official metric is the average Haversine distance in meters.

This notebook is the human-readable companion to the scripts:

- `Img2GPS/extract_exif.py` builds `metadata.csv`
- `Img2GPS/preprocess.py` provides `prepare_data` / `load_raw`
- `Img2GPS/model.py` defines the **MobileNetV3-Small + soft-cluster classifier** with hard-coded target stats and cluster-center buffers
- `Img2GPS/train.py` runs the training loop (K-means cluster head, CE + Haversine loss)
- `Img2GPS/eval_project_a.py` is the course-style evaluator

### Architecture rationale (why we moved off MSE regression)

The original ResNet-18 + 2-D MSE-regression head collapsed to predicting the training centroid (~80 m on the leaderboard, *worse* than the 48.7 m constant-mean baseline). The current setup predicts `K` softmax logits over K-means location clusters (default K=80, configurable via `train.py --num-clusters`) and outputs the weighted average of the cluster centers in raw degrees — making mean-collapse architecturally impossible while still emitting `[lat, lon]` per the spec.


# Colab bootstrap
clone the repo and install dependencies on first run. Safe to re-run locally; it's a no-op when the repo already exists.

In [1]:
import os, sys, subprocess

REPO_URL = "https://github.com/SheilaBkny/cis_5190_project.git"
REPO_BRANCH = "iter2"
COLAB_REPO_DIR = "/content/cis_5190_project"

IN_COLAB = "google.colab" in sys.modules

def _git(*args):
    subprocess.run(["git", "-C", COLAB_REPO_DIR, *args], check=True)

if IN_COLAB:
    if not os.path.exists(os.path.join(COLAB_REPO_DIR, "Img2GPS")):
        subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, COLAB_REPO_DIR],
            check=True,
        )
    else:
        # Repo dir already exists from an earlier session — pull the
        # latest commit on iter1 so new data / notebook fixes show up
        # without needing to delete /content/cis_5190_project manually.
        _git("fetch", "origin", REPO_BRANCH)
        _git("checkout", REPO_BRANCH)
        _git("reset", "--hard", f"origin/{REPO_BRANCH}")
    os.chdir(COLAB_REPO_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True,
    )
    head = subprocess.check_output(["git", "-C", COLAB_REPO_DIR, "log", "-1", "--oneline"]).decode().strip()
    print(f"on commit: {head}")

print("cwd:", os.getcwd())

on commit: 939ac11 k-means 80
cwd: /content/cis_5190_project


In [2]:
import math
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
while REPO_ROOT.name and not (REPO_ROOT / "Img2GPS").exists():
    REPO_ROOT = REPO_ROOT.parent
PROJECT_DIR = REPO_ROOT / "Img2GPS"
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(REPO_ROOT)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("count :", torch.cuda.device_count())
    print("mem GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
print("torch :", torch.__version__)

from preprocess import load_raw, prepare_data  # noqa: E402
from model import Model  # noqa: E402
from train import haversine_meters, location_grouped_split  # noqa: E402

REPO_ROOT, PROJECT_DIR

cuda available: True
device: Tesla T4
count : 1
mem GB: 15.64
torch : 2.10.0+cu128


(PosixPath('/content/cis_5190_project'),
 PosixPath('/content/cis_5190_project/Img2GPS'))

## 1. Data summary

We collected 89 photos around Penn's campus and extracted GPS coordinates from EXIF + Apple location xattrs. The location-grouped split makes sure photos that share an exact GPS coordinate (about 8 photos per spot) live entirely on one side of the train/val split, which avoids leakage.

In [3]:
df = pd.read_csv(PROJECT_DIR / "metadata.csv")
print(f"rows: {len(df)}")
print(f"unique locations: {df[['latitude','longitude']].drop_duplicates().shape[0]}")
df.describe(percentiles=[0.05, 0.5, 0.95]).round(6)

rows: 89
unique locations: 60


,latitude,longitude
count,89.000000,89.000000
mean,39.951564,-75.191324
std,0.000253,0.000552
min,39.951063,-75.192170
5%,39.951187,-75.191850
50%,39.951497,-75.191612
95%,39.952113,-75.190313
max,39.952113,-75.190287


In [4]:
def haversine(a, b):
    R = 6_371_000.0
    lat1, lon1 = map(math.radians, a)
    lat2, lon2 = map(math.radians, b)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    h = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 2 * R * math.asin(math.sqrt(h))

mean_lat = df['latitude'].mean()
mean_lon = df['longitude'].mean()
lat_span = haversine((df['latitude'].min(), mean_lon), (df['latitude'].max(), mean_lon))
lon_span = haversine((mean_lat, df['longitude'].min()), (mean_lat, df['longitude'].max()))
print(f"bounding box: {lat_span:.1f} m (NS) x {lon_span:.1f} m (EW)")
print(f"training mean: ({mean_lat:.6f}, {mean_lon:.6f})")

constant_dist = [haversine((mean_lat, mean_lon), p) for p in df[['latitude','longitude']].values]
print(f"constant-mean baseline Haversine: mean={np.mean(constant_dist):.2f}m  p50={np.median(constant_dist):.2f}m  max={np.max(constant_dist):.2f}m")

bounding box: 116.8 m (NS) x 160.5 m (EW)
training mean: (39.951564, -75.191324)
constant-mean baseline Haversine: mean=48.72m  p50=42.60m  max=107.47m


In [5]:
# Phone-captured images only. The test region is a tiny ~150 x 110 m
# rectangle on Penn's campus, and the spec sampling pattern (a few
# bearings per location) already gives in-domain coverage on this scale.
main_csv = PROJECT_DIR / "metadata.csv"
TRAIN_CSV = str(main_csv)
print(f"phone-only training: {len(pd.read_csv(main_csv))} rows  ({main_csv.name})")
TRAIN_CSV

phone-only training: 89 rows  (metadata.csv)


'/content/cis_5190_project/Img2GPS/metadata.csv'

## 2. Train (single run, no holdout)

Trains on **every example** in `TRAIN_CSV` (phone-only `metadata.csv`, 89 rows by default) using `--use-all-data`. With ~89 images, holding out 18 as a fixed val split throws away meaningful gradient signal — so the simple workflow is to train on all of it and rely on the leaderboard or §2b's bootstrap CV for an out-of-sample estimate.

Under the hood, `train.py`:

1. Runs **K-means with `K=--num-clusters`** (default 80) on the full training set's `(lat, lon)` to produce cluster centers.
2. Stamps the centers onto the model's `cluster_centers` buffer (saved inside `model.pt`). The classifier head is sized to `K` and `K` itself is auto-detected from `model.pt` at load time, so the spec's zero-arg `Model()` works regardless of which `K` you trained.
3. Trains a **soft classifier**: backbone outputs `K` logits, `softmax(logits) @ centers` gives the predicted lat/lon in raw degrees.
4. Optimizes `cross_entropy(logits, closest_cluster) + hav_w * (haversine_m / 1000)` so training is aligned with the eval metric.
5. Freezes everything except the final MobileNetV3-Small block + classifier head — essential with only ~70-90 training images.
6. Photometric-only augmentation (no horizontal flip, no rotation — those break bearing cues).

Saves the final-epoch checkpoint to `Img2GPS/model.pt` and prints a final `RESULT mode=all_data k=… lr=… hav_w=… val_haversine_m=…` line (where `val_haversine_m` here is the train Haversine, since there is no val set). For a real out-of-sample estimate, use §2b's bootstrap CV. Skip this cell if you want to evaluate the existing checkpoint.

In [ ]:
# K=80 in model.py puts us in the instance-retrieval regime (~1-2 train
# images per cluster), so we lean more on the Haversine soft-pred loss
# (--haversine-loss-weight 0.5) since CE class signal becomes noisy.
# --use-all-data trains on every example with no val holdout — see §2b
# for the bootstrap-CV workflow that gives an honest val estimate.
print(f"TRAIN_CSV={TRAIN_CSV}")
!python Img2GPS/train.py --csv "{TRAIN_CSV}" --epochs 30 --lr 1e-3 --haversine-loss-weight 0.5 --use-all-data

## 2b. Hyperparameter sweep with bootstrap CV

With only ~89 phone images, holding out 18 as a fixed val split throws away data we'd rather train on. Instead the sweep cells below use **location-level bootstrap CV** (`train.py --bootstrap-rounds B`):

* Each "round" samples the ~48 unique GPS locations *with replacement*, takes all images at the sampled locations as the training set (a location drawn 3× contributes its images 3× to the gradient), and uses all images at the *unsampled* locations as the out-of-bag (OOB) val set.
* `B` rounds → mean ± std of OOB Haversine. Across rounds, every location ends up on the OOB side at least once, so **no example is permanently held out**.

The plan:

1. Define `run_experiment(...)` — wraps `Img2GPS/train.py`, parses its `RESULT` line, appends to `sweep_results`. Sweep stages use `bootstrap_rounds=3` (B=3) for a quick-but-stable estimate.
2. **Stage 1** — sweep `K` at default `lr` / `hav_w`, ~15 epochs × 3 bootstrap rounds.
3. **Stage 2** — lock `K` to the best, sweep `lr × hav_w`.
4. **Stage 3** — re-train the overall best config for 30 epochs on **every example** (`--use-all-data`) and save it as the leaderboard `Img2GPS/model.pt`.

Sweep runs write to `/tmp/sweep_model.pt`; only Stage 3 overwrites `Img2GPS/model.pt`.

In [ ]:
# Helper: run train.py as a subprocess, parse the trailing RESULT line.
# Each call appends one dict to `sweep_results`. Defaults to bootstrap
# CV with B=3 rounds so val_haversine_m is the mean OOB metric (every
# example contributes across rounds rather than being permanently held
# out).
import re
import subprocess
import time

import pandas as pd

_RESULT_RE = re.compile(r"^RESULT\s+(.+)$", re.MULTILINE)
sweep_results: list[dict] = []


def run_experiment(
    *,
    k: int = 80,
    lr: float = 1e-3,
    hav_w: float = 0.5,
    epochs: int = 15,
    batch_size: int = 32,
    seed: int = 42,
    bootstrap_rounds: int = 3,
    use_all_data: bool = False,
    csv: str = TRAIN_CSV,
    output: str = "/tmp/sweep_model.pt",
    verbose: bool = False,
) -> dict:
    cmd = [
        "python",
        "Img2GPS/train.py",
        "--csv", csv,
        "--output", output,
        "--epochs", str(epochs),
        "--lr", str(lr),
        "--haversine-loss-weight", str(hav_w),
        "--num-clusters", str(k),
        "--batch-size", str(batch_size),
        "--seed", str(seed),
    ]
    if use_all_data:
        cmd.append("--use-all-data")
    elif bootstrap_rounds > 0:
        cmd += ["--bootstrap-rounds", str(bootstrap_rounds)]

    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if verbose:
        print(proc.stdout[-2500:])
    m = _RESULT_RE.search(proc.stdout)
    if m is None:
        print("--- stdout (tail) ---\n", proc.stdout[-2000:])
        print("--- stderr (tail) ---\n", proc.stderr[-2000:])
        raise RuntimeError("training run did not emit a RESULT line")
    fields = dict(part.split("=") for part in m.group(1).split())
    record: dict = {}
    for key, val in fields.items():
        try:
            record[key] = int(val) if "." not in val and "e" not in val.lower() else float(val)
        except ValueError:
            record[key] = val
    record["wall_time_s"] = round(time.time() - t0, 1)
    sweep_results.append(record)
    return record


def results_table() -> pd.DataFrame:
    if not sweep_results:
        return pd.DataFrame()
    cols_priority = [
        "mode", "k", "lr", "hav_w", "epochs", "bootstrap_rounds",
        "val_haversine_m", "val_haversine_m_mean", "val_haversine_m_std",
        "wall_time_s", "batch_size", "weight_decay", "seed",
    ]
    df = pd.DataFrame(sweep_results)
    cols = [c for c in cols_priority if c in df.columns] + [
        c for c in df.columns if c not in cols_priority
    ]
    return df[cols].sort_values("val_haversine_m").reset_index(drop=True)


# Stage 1: sweep K only. B=3 bootstrap rounds × 15 epochs each.
# val_haversine_m in the table is the mean OOB Haversine across rounds.
sweep_results.clear()
for k in [16, 32, 48, 64, 80, 100]:
    rec = run_experiment(k=k, epochs=15, bootstrap_rounds=3)
    print(
        f"K={k:3d}  oob_hav={rec['val_haversine_m_mean']:6.2f} ± "
        f"{rec['val_haversine_m_std']:.2f} m  ({rec['wall_time_s']:.1f}s)"
    )

results_table().head(10)

In [ ]:
# Stage 2: lock K to the Stage-1 winner, sweep (lr, hav_w) with B=3
# bootstrap rounds. Add / change axes (epochs, batch_size, weight_decay
# all flow through run_experiment -> train.py).
from itertools import product

best_k = int(results_table().iloc[0]["k"])
print(f"locked K={best_k}; sweeping lr x hav_w (B=3 bootstrap rounds each)")

for lr, hav_w in product([5e-4, 1e-3, 2e-3], [0.1, 0.5, 1.0, 2.0]):
    rec = run_experiment(k=best_k, lr=lr, hav_w=hav_w, epochs=15, bootstrap_rounds=3)
    print(
        f"  lr={lr:.0e}  hav_w={hav_w:>4}  "
        f"oob_hav={rec['val_haversine_m_mean']:6.2f} ± "
        f"{rec['val_haversine_m_std']:.2f} m"
    )

results_table().head(15)

In [ ]:
# Stage 3: re-train the overall best config with full epochs on EVERY
# example (no holdout) and overwrite Img2GPS/model.pt — the file the
# leaderboard pulls. The bootstrap-CV mean from sweeping is our
# expected leaderboard score; this final run gets a few percent more
# data than any individual bootstrap round, so the leaderboard model
# should match or slightly beat the CV mean.
best = results_table().iloc[0].to_dict()
print("best config (sorted by mean OOB Haversine across bootstrap rounds):")
for key in ("mode", "k", "lr", "hav_w", "epochs", "bootstrap_rounds",
            "val_haversine_m_mean", "val_haversine_m_std", "wall_time_s"):
    if key in best:
        print(f"  {key}: {best[key]}")

final = run_experiment(
    k=int(best["k"]),
    lr=float(best["lr"]),
    hav_w=float(best["hav_w"]),
    epochs=30,
    use_all_data=True,
    output="Img2GPS/model.pt",
    verbose=True,
)
print(
    f"\nfinal model trained on all {89} examples — "
    f"final train_haversine_m = {final['val_haversine_m']:.2f} m "
    f"(CV-estimated leaderboard score ≈ {best['val_haversine_m_mean']:.2f} ± "
    f"{best['val_haversine_m_std']:.2f} m)"
)
print("saved to Img2GPS/model.pt")
results_table().head(15)

## 2.5 Save the trained model to your machine

`Img2GPS/model.pt` only exists inside the Colab VM after training — it is *not* automatically synced back to your laptop or to the `submission` branch. The next cell:

1. Prints the file's size and a short MD5 fingerprint so you can confirm it's the run you just produced.
2. If running in Colab, triggers a browser download.

After the download completes, place the file at `Img2GPS/model.pt` in your local clone and run `Img2GPS/scripts/promote_to_submission.sh` to copy it onto the `submission` branch and push — that's the version the leaderboard pulls.

In [10]:
import hashlib, os, sys

MODEL_PATH = str(PROJECT_DIR / "model.pt")
size_mb = os.path.getsize(MODEL_PATH) / 1e6
md5 = hashlib.md5(open(MODEL_PATH, "rb").read()).hexdigest()[:12]
print(f"model.pt   size: {size_mb:.1f} MB   md5: {md5}")

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(MODEL_PATH)
    print("Downloaded. Move ~/Downloads/model.pt -> <repo>/Img2GPS/model.pt locally,")
    print("then: bash Img2GPS/scripts/promote_to_submission.sh")
else:
    print("Not in Colab — model.pt is already on your local filesystem at the path above.")

model.pt   size: 6.4 MB   md5: 7f3df168327a


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded. Move ~/Downloads/model.pt -> <repo>/Img2GPS/model.pt locally,
then: bash Img2GPS/scripts/promote_to_submission.sh


## 3. Evaluate the saved checkpoint

We load `model.pt` (the best-by-Haversine snapshot) and report the official metrics on:

1. The full collected dataset.
2. The held-out validation split (location-grouped, the honest signal).
3. The course-provided reference set in `Img2GPS/reference/`.

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Model(weights_path=str(PROJECT_DIR / 'model.pt')).to(device).eval()
print(f"y_mean: {model.y_mean.tolist()}")
print(f"y_std:  {model.y_std.tolist()}")
print(f"K cluster centers (lat, lon):")
for i, (lat, lon) in enumerate(model.cluster_centers.tolist()):
    print(f"  [{i:02d}]  ({lat:.6f}, {lon:.6f})")

y_mean: [39.951541900634766, -75.19132232666016]
y_std:  [0.0002309196861460805, 0.0005374249303713441]
K cluster centers (lat, lon):
  [00]  (39.951481, -75.191612)
  [01]  (39.951408, -75.190590)
  [02]  (39.952114, -75.190315)
  [03]  (39.951401, -75.191254)
  [04]  (39.951702, -75.191795)
  [05]  (39.951176, -75.191689)
  [06]  (39.951473, -75.190857)
  [07]  (39.951538, -75.192162)
  [08]  (39.951488, -75.191803)
  [09]  (39.951084, -75.191467)
  [10]  (39.951698, -75.191757)
  [11]  (39.951580, -75.191620)
  [12]  (39.951488, -75.190765)
  [13]  (39.951630, -75.191490)
  [14]  (39.951237, -75.191696)
  [15]  (39.951611, -75.191689)
  [16]  (39.951756, -75.191711)
  [17]  (39.951214, -75.191772)
  [18]  (39.951534, -75.191742)
  [19]  (39.951515, -75.191666)
  [20]  (39.951202, -75.191727)
  [21]  (39.951481, -75.190796)
  [22]  (39.951385, -75.191193)
  [23]  (39.951534, -75.191635)
  [24]  (39.951603, -75.191734)
  [25]  (39.951469, -75.191849)
  [26]  (39.951736, -75.191765)
  

In [12]:
def evaluate(csv_path):
    X, y = prepare_data(str(csv_path))
    with torch.no_grad():
        preds = model(X.to(device)).cpu()
    distances = haversine_meters(preds, y).numpy()
    return preds, y, distances

preds_full, y_full, dists_full = evaluate(PROJECT_DIR / 'metadata.csv')
print(f"FULL set (n={len(y_full)}):")
print(f"  mean={dists_full.mean():.2f}m  p50={np.median(dists_full):.2f}m  p90={np.quantile(dists_full,0.9):.2f}m  max={dists_full.max():.2f}m")

FULL set (n=89):
  mean=35.64m  p50=27.98m  p90=62.21m  max=117.86m


In [13]:
_, y_all = load_raw(str(PROJECT_DIR / 'metadata.csv'))
train_idx, val_idx = location_grouped_split(y_all, val_fraction=0.2, seed=42)
X_full, _ = prepare_data(str(PROJECT_DIR / 'metadata.csv'))
with torch.no_grad():
    preds_val = model(X_full[val_idx].to(device)).cpu()
dists_val = haversine_meters(preds_val, y_all[val_idx]).numpy()
print(f"VAL set (held-out, n={len(val_idx)}):")
print(f"  mean={dists_val.mean():.2f}m  p50={np.median(dists_val):.2f}m  p90={np.quantile(dists_val,0.9):.2f}m  max={dists_val.max():.2f}m")

VAL set (held-out, n=18):
  mean=49.47m  p50=33.57m  p90=116.23m  max=118.42m


In [14]:
ref_csv = PROJECT_DIR / 'reference' / 'metadata.csv'
preds_ref, y_ref, dists_ref = evaluate(ref_csv)
print(f"REFERENCE set (n={len(y_ref)}):")
for (lat, lon), (plat, plon), d in zip(y_ref.tolist(), preds_ref.tolist(), dists_ref):
    print(f"  truth=({lat:.6f},{lon:.6f})  pred=({plat:.6f},{plon:.6f})  haversine={d:.2f}m")
print(f"reference mean Haversine: {dists_ref.mean():.2f}m")

REFERENCE set (n=6):
  truth=(39.952309,-75.191574)  pred=(39.951591,-75.191544)  haversine=79.79m
  truth=(39.952324,-75.191582)  pred=(39.951580,-75.191513)  haversine=82.92m
  truth=(39.952301,-75.191551)  pred=(39.951630,-75.191559)  haversine=74.66m
  truth=(39.952301,-75.191551)  pred=(39.951561,-75.191360)  haversine=83.88m
  truth=(39.952309,-75.191574)  pred=(39.951580,-75.191429)  haversine=81.95m
  truth=(39.952301,-75.191551)  pred=(39.951550,-75.191490)  haversine=83.72m
reference mean Haversine: 81.15m
